<a href="https://colab.research.google.com/github/EvenSol/NeqSim-Colab/blob/master/notebooks/reactions/natural_gas_combustion_with_cantera.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Natural-gas combustion and air blending with Cantera

This engineering example calculates combustion products for natural gases of different compositions while varying the air-to-fuel ratio. It uses [Cantera](https://cantera.org/) and the detailed `gri30.yaml` natural-gas combustion mechanism.

The notebook answers practical screening questions:

* How much stoichiometric air is required by each gas composition?
* How do excess air, fuel richness, hydrogen blending, CO2 dilution, and oxidizer composition affect adiabatic flame temperature?
* What are the wet and dry flue-gas concentrations of CO2, H2O, O2, CO, H2, unburned CH4, and equilibrium NOx?
* How can a composition prepared in NeqSim be mapped into a combustion calculation?

The central variable is the excess-air ratio

$$\lambda = \frac{(air/fuel)_{actual}}{(air/fuel)_{stoichiometric}} = \frac{1}{\phi},$$

where $\phi$ is Cantera's equivalence ratio. Thus $\lambda<1$ is fuel-rich, $\lambda=1$ is stoichiometric, and $\lambda>1$ is fuel-lean (excess air).

## Model scope

Cantera provides thermodynamics, chemical kinetics, and transport models for reacting systems. Here we first use **adiabatic, constant-pressure chemical equilibrium** (`equilibrate("HP")`). Enthalpy and pressure are held constant while the product composition and temperature are solved.

This is an excellent first-pass heat-and-material-balance model, but it is not a burner performance guarantee. It assumes complete mixing, no heat loss, and enough time to reach equilibrium. Later sections explain why equilibrium CO and NOx must not be treated as stack-emission predictions.

In [ ]:
# Install only when needed (for example, in a fresh Google Colab runtime).
import importlib.util
import subprocess
import sys

if importlib.util.find_spec("cantera") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "cantera>=3.1"])

In [ ]:
import cantera as ct
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

print(f"Cantera version: {ct.__version__}")
gas_check = ct.Solution("gri30.yaml")
print(f"Mechanism: {gas_check.source}; {gas_check.n_species} species; {gas_check.n_reactions} reactions")

## Fuel definitions

The examples are normalized mole fractions. They deliberately cover lean pipeline gas, a richer gas, CO2-diluted gas, and a 20 mol% hydrogen blend. `gri30.yaml` includes methane, ethane, and propane but is not a general mechanism for arbitrary C4+ natural gas, sulfur species, soot, or ammonia. Use a validated mechanism containing every required fuel and pollutant species for those cases.

The oxidizer below represents dry atmospheric air. Separate oxidizer-blending cases are introduced later.

In [ ]:
def normalize(composition):
    total = sum(composition.values())
    if total <= 0:
        raise ValueError("Composition total must be positive")
    return {name: value / total for name, value in composition.items() if value > 0}


FUELS = {
    "Lean pipeline gas": normalize({"CH4": 0.93, "C2H6": 0.04, "C3H8": 0.01, "CO2": 0.01, "N2": 0.01}),
    "Rich natural gas": normalize({"CH4": 0.82, "C2H6": 0.10, "C3H8": 0.04, "CO2": 0.02, "N2": 0.02}),
    "CO2-diluted gas": normalize({"CH4": 0.88, "CO2": 0.10, "N2": 0.02}),
    "20% H2 blend": normalize({"CH4": 0.76, "H2": 0.20, "C2H6": 0.02, "CO2": 0.01, "N2": 0.01}),
}

DRY_AIR = normalize({"O2": 0.2095, "N2": 0.7808, "AR": 0.0093, "CO2": 0.0004})

fuel_table = pd.DataFrame(FUELS).fillna(0.0).T * 100
fuel_table.index.name = "Fuel [mol%]"
display(fuel_table.round(2))

## Stoichiometric demand and carbon intensity

For one mole of a species $C_xH_yO_z$, complete combustion requires

$$\nu_{O_2}=x+\frac{y}{4}-\frac{z}{2}.$$

The calculation below applies the elemental balance to the complete mixture. The reported CO2 factor is the maximum stack CO2 if all feed carbon exits as CO2. The separate “formed” factor excludes CO2 already present in the fuel. Whether feed CO2 is counted in an emissions inventory depends on its origin and the applicable reporting boundary.

In [ ]:
def fuel_balance_metrics(fuel):
    mechanism = ct.Solution("gri30.yaml")
    fuel = normalize(fuel)
    atoms = {element: 0.0 for element in ("C", "H", "O")}
    molar_mass = 0.0  # kg/kmol
    for species, fraction in fuel.items():
        k = mechanism.species_index(species)
        molar_mass += fraction * mechanism.molecular_weights[k]
        for element in atoms:
            atoms[element] += fraction * mechanism.n_atoms(species, element)

    o2_stoich = atoms["C"] + atoms["H"] / 4.0 - atoms["O"] / 2.0
    feed_co2 = fuel.get("CO2", 0.0)
    mw_co2 = mechanism.molecular_weights[mechanism.species_index("CO2")]
    return {
        "fuel MW [kg/kmol]": molar_mass,
        "O2 stoich [kmol/kmol fuel]": o2_stoich,
        "dry air stoich [kmol/kmol fuel]": o2_stoich / DRY_AIR["O2"],
        "carbon [kmol C/kmol fuel]": atoms["C"],
        "max stack CO2 [kg/kg fuel]": atoms["C"] * mw_co2 / molar_mass,
        "CO2 formed [kg/kg fuel]": (atoms["C"] - feed_co2) * mw_co2 / molar_mass,
    }


fuel_metrics = pd.DataFrame({name: fuel_balance_metrics(comp) for name, comp in FUELS.items()}).T
display(fuel_metrics.round(4))

## Reusable equilibrium-combustion calculation

The function uses Cantera's `set_equivalence_ratio` to blend fuel and oxidizer at a requested $\lambda$, then equilibrates the mixture at constant enthalpy and pressure. Results are reported on both wet and dry bases. Dry ppm values exclude water, matching common flue-gas reporting practice.

The NOx value is the equilibrium sum of NO and NO2. The final column applies the common oxygen-reference conversion to 15 vol% O2 on a dry basis. Confirm the required reference oxygen and regulatory convention before using that conversion in a real report.

In [ ]:
def run_combustion(fuel_name, fuel, lambda_air, oxidizer=DRY_AIR,
                   oxidizer_name="Dry air", inlet_temperature_K=288.15,
                   pressure_bar=1.01325):
    if lambda_air <= 0:
        raise ValueError("lambda_air must be positive")

    gas = ct.Solution("gri30.yaml")
    gas.TP = inlet_temperature_K, pressure_bar * 1e5
    gas.set_equivalence_ratio(
        phi=1.0 / lambda_air,
        fuel=normalize(fuel),
        oxidizer=normalize(oxidizer),
        basis="mole",
    )
    gas.equilibrate("HP")

    x = dict(zip(gas.species_names, gas.X))
    dry_total = max(1.0 - x.get("H2O", 0.0), 1e-30)
    dry = lambda species: x.get(species, 0.0) / dry_total
    o2_dry_pct = 100.0 * dry("O2")
    nox_dry_ppm = 1e6 * (dry("NO") + dry("NO2"))
    correction_15 = ((20.9 - 15.0) / (20.9 - o2_dry_pct)) if o2_dry_pct < 20.9 else np.nan

    return {
        "fuel": fuel_name,
        "oxidizer": oxidizer_name,
        "lambda": lambda_air,
        "excess air [%]": 100.0 * (lambda_air - 1.0),
        "T adiabatic [degC]": gas.T - 273.15,
        "H2O wet [vol%]": 100.0 * x.get("H2O", 0.0),
        "CO2 dry [vol%]": 100.0 * dry("CO2"),
        "O2 dry [vol%]": o2_dry_pct,
        "CO dry [ppmv]": 1e6 * dry("CO"),
        "NOx dry [ppmv]": nox_dry_ppm,
        "NOx dry @15% O2 [ppmv]": nox_dry_ppm * correction_15,
        "CH4 dry [ppmv]": 1e6 * dry("CH4"),
        "H2 dry [vol%]": 100.0 * dry("H2"),
    }


reference = run_combustion("Lean pipeline gas", FUELS["Lean pipeline gas"], lambda_air=1.15)
display(pd.DataFrame([reference]).set_index(["fuel", "oxidizer"]).round(3))

## Sensitivity to fuel composition and air-to-fuel ratio

The sweep spans rich operation ($\lambda=0.8$), stoichiometric operation, and 5–100% excess air. In normal industrial combustion, the useful operating window is constrained by flame stability, CO, NOx, temperature, efficiency, equipment design, and safety systems; it is not selected from equilibrium results alone.

In [ ]:
lambdas = [0.80, 1.00, 1.05, 1.15, 1.30, 1.60, 2.00]
results = pd.DataFrame([
    run_combustion(name, composition, lambda_air)
    for name, composition in FUELS.items()
    for lambda_air in lambdas
])

summary_columns = [
    "fuel", "lambda", "T adiabatic [degC]", "H2O wet [vol%]",
    "CO2 dry [vol%]", "O2 dry [vol%]", "CO dry [ppmv]",
    "NOx dry [ppmv]", "CH4 dry [ppmv]", "H2 dry [vol%]",
]
display(results[summary_columns].round(3))

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 9), sharex=True)
plots = [
    ("T adiabatic [degC]", "Adiabatic temperature [degC]", False),
    ("CO2 dry [vol%]", "Dry CO2 [vol%]", False),
    ("O2 dry [vol%]", "Dry O2 [vol%]", False),
    ("CO dry [ppmv]", "Equilibrium dry CO [ppmv]", True),
]

for fuel_name, group in results.groupby("fuel"):
    for ax, (column, ylabel, log_scale) in zip(axes.flat, plots):
        ax.plot(group["lambda"], group[column].clip(lower=1e-6), marker="o", label=fuel_name)
        ax.set_ylabel(ylabel)
        ax.grid(True, alpha=0.3)
        if log_scale:
            ax.set_yscale("log")

for ax in axes[-1, :]:
    ax.set_xlabel("Excess-air ratio, lambda [-]")
axes[0, 0].legend(fontsize=8)
fig.suptitle("Fuel composition and air blending: adiabatic equilibrium screening")
fig.tight_layout()
plt.show()

### Reading the trends

* Near-stoichiometric combustion gives the highest adiabatic temperature. Both excess air and fuel-rich operation reduce it.
* More excess air dilutes CO2 and H2O, raises residual O2, and increases the mass/volume of flue gas that equipment must handle.
* Rich operation produces substantial equilibrium CO and H2 because there is insufficient oxygen for complete conversion.
* CO2-diluted fuel has a lower flame temperature because inert material absorbs heat. Its total stack CO2 includes CO2 entering with the fuel.
* Hydrogen blending changes stoichiometric demand, water production, temperature, and CO2 per unit fuel. Compare on an energy basis—not only per kg or per kmol—when assessing decarbonization.
* Equilibrium thermal NO generally rises strongly with flame temperature and available oxygen. Actual NOx is controlled by finite-rate chemistry, residence time, mixing, pressure, heat loss, burner geometry, and staging.

## Changing the oxidizer blend

“Air blending” can mean changing the amount of air, represented by $\lambda$, or changing the oxidizer composition. The cases below compare dry air, humid air, an illustrative 10% exhaust-gas-recirculation (EGR) blend, and oxygen-enriched air at the same $\lambda=1.15$.

The EGR composition is only a screening proxy. In a plant model, recycle composition should be taken from the calculated or measured flue gas and solved iteratively with the burner and recycle loop.

In [ ]:
OXIDIZERS = {
    "Dry air": DRY_AIR,
    "2% humid air": normalize({**{k: 0.98 * v for k, v in DRY_AIR.items()}, "H2O": 0.02}),
    "10% EGR proxy": normalize({
        "O2": 0.90 * DRY_AIR["O2"],
        "N2": 0.90 * DRY_AIR["N2"],
        "AR": 0.90 * DRY_AIR["AR"],
        "CO2": 0.90 * DRY_AIR["CO2"] + 0.07,
        "H2O": 0.03,
    }),
    "25% O2 enriched": normalize({"O2": 0.25, "N2": 0.75}),
}

oxidizer_results = pd.DataFrame([
    run_combustion(
        "Lean pipeline gas", FUELS["Lean pipeline gas"], lambda_air=1.15,
        oxidizer=composition, oxidizer_name=name,
    )
    for name, composition in OXIDIZERS.items()
])

display(oxidizer_results[[
    "oxidizer", "T adiabatic [degC]", "H2O wet [vol%]", "CO2 dry [vol%]",
    "O2 dry [vol%]", "CO dry [ppmv]", "NOx dry [ppmv]",
]].set_index("oxidizer").round(3))

Humid air and EGR add heat capacity and normally lower flame temperature. Oxygen enrichment reduces nitrogen dilution and can strongly increase temperature, so materials limits and burner design become critical. Comparing cases at equal $\lambda$ does not imply equal total oxidizer flow, oxygen concentration, flame speed, or equipment duty.

## Bringing a NeqSim composition into Cantera

NeqSim is well suited to upstream PVT and process simulation: separation, compression, fuel-gas conditioning, condensation, and stream properties. Cantera then adds detailed reacting-system calculations. A practical workflow is:

1. Run the fuel-gas conditioning/process case in NeqSim.
2. Extract the gas-phase mole fractions at the burner boundary.
3. Map component names and retain only species supported by the selected Cantera mechanism.
4. Normalize, document any C4+ lumping, and run the combustion sensitivity.
5. Return flue-gas flow, composition, temperature, and heat duty to the wider process/emissions study.

The helper below demonstrates the name mapping. It fails visibly for unsupported nonzero species rather than silently dropping carbon.

In [ ]:
NEQSIM_TO_CANTERA = {
    "methane": "CH4", "ethane": "C2H6", "propane": "C3H8",
    "hydrogen": "H2", "CO2": "CO2", "nitrogen": "N2",
    "oxygen": "O2", "water": "H2O",
}


def map_neqsim_gas_to_cantera(neqsim_mole_fractions, tolerance=1e-12):
    mapped = {}
    unsupported = {}
    for name, fraction in neqsim_mole_fractions.items():
        if fraction <= tolerance:
            continue
        cantera_name = NEQSIM_TO_CANTERA.get(name)
        if cantera_name is None:
            unsupported[name] = fraction
        else:
            mapped[cantera_name] = mapped.get(cantera_name, 0.0) + fraction
    if unsupported:
        raise ValueError(
            "Unsupported nonzero species; choose a suitable mechanism or an explicit lumping rule: "
            + str(unsupported)
        )
    return normalize(mapped)


neqsim_example = {
    "methane": 0.93, "ethane": 0.04, "propane": 0.01,
    "CO2": 0.01, "nitrogen": 0.01,
}
mapped_fuel = map_neqsim_gas_to_cantera(neqsim_example)
display(pd.Series(mapped_fuel, name="Cantera mole fraction").to_frame())

## Engineering use and limitations

This notebook can support concept selection, burner/furnace heat-and-material balances, gas-turbine fuel sensitivity, flue-gas equipment screening, CO2 capture inlet estimates, oxygen-analyzer cross-checks, EGR/oxygen-enrichment studies, and scenario generation for emissions work.

For an engineering study, extend the result object with fuel flow and heating value, calculate wet and dry flue-gas flow rates, close the heat balance against a specified furnace duty or turbine efficiency, and propagate uncertainty in fuel analysis, humidity, leakage air, and measurement error.

Important limitations:

* **Temperature:** adiabatic equilibrium temperature is an upper-bound reference, not a measured flame, turbine-inlet, furnace-exit, or stack temperature.
* **CO and unburned hydrocarbons:** equilibrium does not represent quenching, incomplete mixing, wall effects, flame instability, or finite residence time.
* **NOx:** equilibrium NOx is not a regulatory emissions prediction. Use a validated finite-rate reactor/flame model and burner-specific data for design or compliance work.
* **Mechanism validity:** verify that the mechanism contains and has been validated for the actual fuel species, pressure, temperature, dilution, and equivalence-ratio range.
* **Trace species:** sulfur, particulates, soot, metals, and many fuel contaminants are outside `gri30.yaml`.
* **Basis:** always label wet/dry basis, ppm/vol%, reference O2, pressure, inlet temperature, and whether CO2 already present in the fuel is counted.

## References and next steps

* [Cantera home page](https://cantera.org/)
* [Cantera Python tutorial](https://cantera.org/stable/userguide/python-tutorial.html)
* [Cantera example gallery](https://cantera.org/stable/examples/)

Natural next steps are a constant-pressure stirred-reactor model for residence-time effects, an ignition-delay study, or a one-dimensional premixed-flame model for flame speed and spatial temperature/species profiles.